# Evaluation Notebook : Multimodal Conversational RAG

**Test document:** *Attention Is All You Need* (Vaswani et al., 2017)

This notebook runs the evaluation question set through the RAG pipeline, scores the results using RAGAS (Faithfulness, Answer Relevance, Context Precision), measures end-to-end latency, and reports a per-category breakdown with explicit failure analysis.

**Prerequisites before running:**
- This notebook must be placed in the project root (same folder as `config.py`, `conversational_RAG.py`, `vector_store.py`, etc.)
- `.env` must contain a valid `GROQ_API_KEY`
- `Attention.pdf` must already be ingested into the vector store (run `python app.py` once beforehand, or run the ingestion cell below)
- `eval_dataset_assigned.json` (or `eval_dataset.py`) must be present with the test questions

## 1. Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import time
import json
from collections import defaultdict

import conversational_RAG
from conversational_RAG import ask_question
from retrieval_profiles import build_retrieval_profile
from vector_store import get_all_documents

print("Setup complete.")

## 2. Confirm the vector store is populated

If this prints 0, the test document hasn't been ingested yet — run `python app.py` from the project root first, then re-run this cell.

In [ ]:
num_chunks = len(get_all_documents())
print(f"Chunks currently in vector store: {num_chunks}")
assert num_chunks > 0, "Vector store is empty — ingest Attention.pdf before continuing."

## 3. Load the evaluation question set

Loads from `eval_dataset_assigned.json` if present (team-split version with `assigned_to` fields); otherwise falls back to the full `eval_dataset.py` question list.

In [ ]:
import os

if os.path.exists("eval_dataset_assigned.json"):
    with open("eval_dataset_assigned.json") as f:
        eval_questions = json.load(f)
    print(f"Loaded {len(eval_questions)} questions from eval_dataset_assigned.json")
else:
    from eval_dataset import eval_questions
    print(f"Loaded {len(eval_questions)} questions from eval_dataset.py")

eval_questions[0]

## 4. Run the pipeline on every question

For each question: reset conversation history (these are independent questions, not a multi-turn conversation), run `ask_question()` with a low-variation retrieval profile to conserve API quota, capture the answer, retrieved contexts, and latency.

In [ ]:
eval_profile = build_retrieval_profile(directness=1.0, semanticness=0.5)

results = []

for item in eval_questions:
    conversational_RAG.chat_history = []  # reset — each eval question is independent
    print(f"Running: {item['question']}")

    start = time.perf_counter()
    answer, source_pages, contexts = ask_question(
        item["question"], retrieval_profile=eval_profile, return_contexts=True
    )
    latency_ms = (time.perf_counter() - start) * 1000

    results.append({
        "question": item["question"],
        "answer": answer,
        "contexts": contexts,
        "ground_truth": item["ground_truth"],
        "category": item["category"],
        "latency_ms": latency_ms,
    })

with open("eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nCompleted {len(results)} questions. Saved to eval_results.json")

### (Optional) Load pre-computed / merged team results instead

If you're scoring the combined results from all team members rather than re-running the pipeline here, skip cell 4 above and run this cell instead.

In [ ]:
# Uncomment to load merged team results instead of re-running the pipeline:
# with open("eval_results_combined.json") as f:
#     results = json.load(f)
# print(f"Loaded {len(results)} pre-computed results")

## 5. Score with RAGAS (judge model: Groq, matching the pipeline's own LLM)

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from config import LLM_MODEL_NAME, EMBEDDING_MODEL_NAME

judge_llm = LangchainLLMWrapper(ChatGroq(model=LLM_MODEL_NAME, temperature=0))
judge_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME))

dataset = Dataset.from_list([
    {
        "question": r["question"],
        "answer": r["answer"],
        "contexts": r["contexts"],
        "ground_truth": r["ground_truth"],
    }
    for r in results
])

scores = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=judge_llm,
    embeddings=judge_embeddings,
)

scores

## 6. Latency summary

In [ ]:
latencies = [r["latency_ms"] for r in results]
avg_latency = sum(latencies) / len(latencies)

print(f"Average latency: {avg_latency:.0f} ms")
print(f"Min latency:     {min(latencies):.0f} ms")
print(f"Max latency:     {max(latencies):.0f} ms")

## 7. Per-category breakdown

Aggregates the RAGAS per-question scores by question category, so we can see whether the pipeline is meaningfully weaker on any particular question type (e.g. table-derived vs. direct factual vs. adversarial).

In [ ]:
scores_df = scores.to_pandas()
scores_df["category"] = [r["category"] for r in results]
scores_df["latency_ms"] = latencies

category_summary = scores_df.groupby("category")[["faithfulness", "answer_relevancy", "context_precision", "latency_ms"]].mean()
category_summary

## 8. Failure analysis — lowest-scoring questions

Explicit, honest inspection of where the pipeline performed worst, per metric — required for the evaluation to be more than just an inflated set of averages.

In [ ]:
for metric in ["faithfulness", "answer_relevancy", "context_precision"]:
    print(f"\n=== Lowest-scoring questions: {metric} ===")
    worst = scores_df.nsmallest(3, metric)
    for _, row in worst.iterrows():
        print(f"  [{row[metric]:.3f}] {row['user_input'] if 'user_input' in row else row.get('question', '?')}")

## 9. Full results table (for appendix / raw data reference)

In [ ]:
scores_df